# Customer Engagement & Product Utilization Analytics for Retention Strategy
**Client:** European Central Bank  
**Phase:** 1 — Data Ingestion & Validation  
**Dataset:** European Bank Customer Churn Dataset (10,000 records)

---
## 1. Imports

In [ ]:
import pandas as pd
import numpy as np

---
## 2. Load Dataset

In [140]:
df = pd.read_csv("../European_Bank.csv")

# Preview first 5 rows
df.head()

,Year,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,2025,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2025,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,2025,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,2025,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,2025,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


---
## 3. Initial Inspection

### 3.1 Shape & Data Types

In [144]:
# Dataset dimensions and column data types
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 14 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Year             10000 non-null  int64  
 1   CustomerId       10000 non-null  int64  
 2   Surname          10000 non-null  object 
 3   CreditScore      10000 non-null  int64  
 4   Geography        10000 non-null  object 
 5   Gender           10000 non-null  object 
 6   Age              10000 non-null  int64  
 7   Tenure           10000 non-null  int64  
 8   Balance          10000 non-null  float64
 9   NumOfProducts    10000 non-null  int64  
 10  HasCrCard        10000 non-null  int64  
 11  IsActiveMember   10000 non-null  int64  
 12  EstimatedSalary  10000 non-null  float64
 13  Exited           10000 non-null  int64  
dtypes: float64(2), int64(9), object(3)
memory usage: 1.1+ MB


### 3.2 Descriptive Statistics

In [147]:
# Summary statistics for all numerical columns
df.describe()

,Year,CustomerId,CreditScore,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
count,10000.0,1.000000e+04,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000,10000.00000,10000.000000,10000.000000,10000.000000
mean,2025.0,1.569094e+07,650.528800,38.921800,5.012800,76485.889288,1.530200,0.70550,0.515100,100090.239881,0.203700
std,0.0,7.193619e+04,96.653299,10.487806,2.892174,62397.405202,0.581654,0.45584,0.499797,57510.492818,0.402769
min,2025.0,1.556570e+07,350.000000,18.000000,0.000000,0.000000,1.000000,0.00000,0.000000,11.580000,0.000000
25%,2025.0,1.562853e+07,584.000000,32.000000,3.000000,0.000000,1.000000,0.00000,0.000000,51002.110000,0.000000
50%,2025.0,1.569074e+07,652.000000,37.000000,5.000000,97198.540000,1.000000,1.00000,1.000000,100193.915000,0.000000
75%,2025.0,1.575323e+07,718.000000,44.000000,7.000000,127644.240000,2.000000,1.00000,1.000000,149388.247500,0.000000
max,2025.0,1.581569e+07,850.000000,92.000000,10.000000,250898.090000,4.000000,1.00000,1.000000,199992.480000,1.000000


---
## 4. Data Cleaning

### 4.1 Duplicate Check

In [151]:
# Check for fully duplicate rows
print('Duplicate rows:', df.duplicated().sum())

# Check for duplicate customer IDs
print('Duplicate CustomerIds:', df['CustomerId'].duplicated().sum())

Duplicate rows: 0
Duplicate CustomerIds: 0


### 4.2 Missing Values

In [154]:
# Check for missing values across all columns
df.isnull().sum()

Year               0
CustomerId         0
Surname            0
CreditScore        0
Geography          0
Gender             0
Age                0
Tenure             0
Balance            0
NumOfProducts      0
HasCrCard          0
IsActiveMember     0
EstimatedSalary    0
Exited             0
dtype: int64

### 4.3 Drop Irrelevant Columns
- **`Year`**: Single value (2025) across all rows — no analytical value  
- **`Surname`**: Identifier only — not used in any analytical phase

In [157]:
df.drop(columns=['Year', 'Surname'], inplace=True)

print('Remaining columns:', df.columns.tolist())
print('Shape after drop:', df.shape)

Remaining columns: ['CustomerId', 'CreditScore', 'Geography', 'Gender', 'Age', 'Tenure', 'Balance', 'NumOfProducts', 'HasCrCard', 'IsActiveMember', 'EstimatedSalary', 'Exited']
Shape after drop: (10000, 12)


---
## 5. Field Validation

### 5.1 Binary Variables Consistency
The following columns must contain only 0 or 1.

In [159]:
# Validate binary fields — expected values: [0, 1] only
print('HasCrCard:     ', df['HasCrCard'].unique())
print('IsActiveMember:', df['IsActiveMember'].unique())
print('Exited:        ', df['Exited'].unique())

HasCrCard:      [1 0]
IsActiveMember: [1 0]
Exited:         [1 0]


### 5.2 Engagement & Product Fields
- **`Tenure`**: Expected range 0–10 years  
- **`NumOfProducts`**: Expected range 1–4

In [162]:
# Validate engagement and product fields
print('Tenure values:         ', sorted(df['Tenure'].unique()))
print('NumOfProducts values:  ', sorted(df['NumOfProducts'].unique()))

Tenure values:          [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
NumOfProducts values:   [1, 2, 3, 4]


---
## 6. Churn Label Verification

In [165]:
# Distribution of the target variable
print(df['Exited'].value_counts())
print()
print(f'Overall Churn Rate: {round(df["Exited"].mean() * 100, 2)}%')

Exited
0    7963
1    2037
Name: count, dtype: int64

Overall Churn Rate: 20.37%


**Interpretation:**
- Churn rate of ~20.37% is realistic for a retail banking context
- Both classes (retained / churned) are present — no labelling errors
- Mild class imbalance — noted but not addressed at this stage